In [2]:
import chromadb
from sentence_transformers import SentenceTransformer
import os

In [3]:
# Create persistent directory
persist_directory = "../chroma_db"

client = chromadb.PersistentClient(path=persist_directory)

collection = client.get_or_create_collection(
    name="dynatrace_public_docs"
)

print("Collection created successfully.")

Collection created successfully.


In [4]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [5]:
docs = [
    "Dynatrace OneAgent installation requires downloading the installer from the Dynatrace environment and running it with proper permissions.",
    "Dynatrace Kubernetes monitoring involves deploying the OneAgent as a DaemonSet in the cluster.",
    "Release notes indicate improvements in log ingestion performance and new OpenTelemetry tracing support."
]

metadatas = [
    {"url": "https://docs.dynatrace.com/oneagent", "source": "docs"},
    {"url": "https://docs.dynatrace.com/kubernetes", "source": "docs"},
    {"url": "https://docs.dynatrace.com/release-notes", "source": "release_notes"}
]

ids = ["doc1", "doc2", "doc3"]

embeddings = model.encode(docs).tolist()

collection.add(
    documents=docs,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings
)

print("Documents added successfully.")

Documents added successfully.


In [6]:
query = "How do I install OneAgent?"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)

print("Top Results:")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print("\n---")
    print("Document:", doc)
    print("Source:", meta["url"])

Top Results:

---
Document: Dynatrace OneAgent installation requires downloading the installer from the Dynatrace environment and running it with proper permissions.
Source: https://docs.dynatrace.com/oneagent

---
Document: Dynatrace Kubernetes monitoring involves deploying the OneAgent as a DaemonSet in the cluster.
Source: https://docs.dynatrace.com/kubernetes
